In [74]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt
from utils import is_casenum, extract_json

rng = np.random.default_rng()


In [75]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 4000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 500
REQUESTS_PER_BATCH = 1000

In [76]:
MIN_IDX = 0
MAX_IDX = 10
REPLACE = False
REPLACE_OUTPUT_FILE = True
MODE = 'WRITE'  # ESTIMATE | BATCH | WRITE

In [77]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [78]:
idx = rng.integers(low=MIN_IDX, high=MAX_IDX+1, size=1)
date = meetings_df.iloc[idx]['date'].values[0]
year = date[0:4]

agenda_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized.json")
agenda_override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-override.json")
if os.path.exists(agenda_override_filepath):
    with open(agenda_override_filepath, 'r', encoding='utf-8') as f:
        agenda_json = json.load(f)
else:
    with open(agenda_filepath, 'r', encoding='utf-8') as f:
        agenda_json = json.load(f)

docs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs.pkl")
docs_df = pd.read_pickle(docs_filepath)

suppdocs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs-summarized.json")
with open(suppdocs_filepath, 'r') as f:
    suppdocs_json = json.load(f)

In [80]:
doc = rng.choice(suppdocs_json)
doc_id = doc['doc_id']
content = docs_df[docs_df['doc_id'] == doc_id]['content'].values[0]
agenda_text = ""
for agenda_item in agenda_json:
    if agenda_item['item_no'] in doc['referenced_agenda_items']:
        agenda_text += f"{agenda_item['text']}\n\n"

TEXT = f"""

DATE: {date}
DOC_ID: {doc_id}

SUMMARY:
{doc['summary']}

DOC_TYPE: {doc['document_type']}
AUTHOR_TYPE: {doc['author_type']}
REFERENCED_ITEMS: {str(doc['referenced_agenda_items'])}

SUPPORT_OR_OPPOSE:
{json.dumps(doc['support_or_oppose'], indent=2)}

=======================================================

DOCUMENT CONTENT:

{content}


========================================================

AGENDA CONTENT:

{agenda_text}

"""

with open("debug.txt", 'w', encoding='utf-8') as f:
    f.write(TEXT)

print(TEXT)



DATE: 2018-07-26
DOC_ID: 18

SUMMARY:
A letter from Jon Dearing, sent via Abundant Housing LA's advocacy tool, expressing support for two housing projects on the agenda. The author urges the Commission to reject the appeal for the mixed-use development at 8521 S Sepulveda Blvd (Item 6) and to approve the General Plan Amendment, zone change, density bonus, and site plan review for the residential project at Lankershim Blvd (Item 7), citing the need for housing and affordable units.

DOC_TYPE: PUBLIC COMMENT
AUTHOR_TYPE: INDIVIDUAL
REFERENCED_ITEMS: ['6', '7']

SUPPORT_OR_OPPOSE:
{
  "6": "DEFINITELY SUPPORT",
  "7": "DEFINITELY SUPPORT"
}


DOCUMENT CONTENT:


                                                                                    
    7/24/2018 City of Los Angeles Mail - Support for projects in Culver City and the Valley, cases DIR-2017-1735-TOC-SPR-CDO-1A & ENV-2017-17361-CE, …
                                                                                    
         